# J3-05 · Deep Agents, le harnais tout fait · Bilan du jour

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 3 · créneau 14h50–15h05 · ⏱️ 15 min (**trace enregistrée** du Deep Agent commentée, bilan ; quiz en autonomie)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Reconnaître dans **Deep Agents** ce que vous avez construit à la main (planification, sous-agents, fichiers) · 2. Mesurer son coût sur un modèle local · 3. Assembler le **jalon J3** et suivre une conversation dans Langfuse · 4. Vérifier vos acquis |
| 🧱 **Fil rouge** | `helpdesk_ops.jalons.j3` : garde-fou + coordinateur + 4 spécialistes, dont le Diagnostic augmenté par MCP |
| 📦 **Versions** | `deepagents 0.7.19` (démonstration ; version compatible avec le lock gelé) |

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

## 2. Deep Agents : ce que vous avez construit, en version harnais

### [Théorie] Un `create_agent` avec des middlewares pré-installés

`create_deep_agent` (paquet `deepagents`, édité par LangChain) renvoie un graphe LangGraph compilé, comme `create_agent`, mais avec des capacités **déjà branchées** :

| Capacité Deep Agents | Ce que vous avez fait à la main |
|---|---|
| outil `task` : des **sous-agents** au contexte isolé | `comme_outil` + `demander_<nom>` (J3-02) |
| outils `ls`, `read_file`, `write_file`, `edit_file`, `delete`, `glob`, `grep` : un **système de fichiers virtuel** dans l'état | l'état LangGraph et ses checkpoints (J2) |
| outil `execute` : commandes shell | rien, et c'est voulu : sans *backend* bac à sable, il renvoie une erreur |
| un **plan** tenu à jour (`write_todos`) | le coordinateur qui décompose la demande ; ⚠️ **optionnel** depuis la 0.7.0 : absent par défaut en 0.7.19 |

> ⚠️ **Deux pièges vérifiés dans la version 0.7.19 :**
> - sans `model=`, Deep Agents utilise par défaut un **modèle cloud** (Claude) : fatal pour le contrat 100 % local. Passez **toujours** le modèle explicitement ;
> - le harnais ajoute un **long prompt système** et une dizaine d'outils : c'est beaucoup de contexte pour un modèle de 8 milliards de paramètres. On le mesure ci-dessous.

In [ ]:
import time

from deepagents import create_deep_agent

from helpdesk_ops.multi_agents import outils_kb, recherche_qdrant
from helpdesk_ops.outils import statut_serveur
from helpdesk_ops.reseau import ping_serveur

profond = create_deep_agent(
    model=llm,                                              # ← explicite, toujours
    tools=outils_kb(recherche_qdrant),
    system_prompt="Tu es l'assistant support N1 de NordTech SA. Réponds en français, en citant les références [KB-xxx].",
    subagents=[{
        "name": "diagnostic",
        "description": "Établit les faits sur un serveur : statut et joignabilité réseau.",
        "system_prompt": "Tu es le spécialiste Diagnostic. Utilise tes outils et rends les faits en 2 phrases.",
        "tools": [statut_serveur, ping_serveur],
    }],
    name="deep-helpdesk",
)
print("Outils vus par le modèle :", sorted(profond.nodes["tools"].bound.tools_by_name))

In [ ]:
debut = time.perf_counter()
with propagate_attributes(session_id="j3-05-deep-agents", tags=["jour-3", "deep-agents"]):
    etat = await profond.ainvoke(
        {"messages": [("user", "srv-erp-02 répond-il au ping ? Et quelle procédure suivre si l'ERP est indisponible ?")]},
        config={"callbacks": [langfuse_handler], "run_name": "deep-agent", "recursion_limit": 40})
duree = time.perf_counter() - debut

appels = [c["name"] for m in etat["messages"] for c in (getattr(m, "tool_calls", None) or [])]
jetons = [m.usage_metadata["input_tokens"] for m in etat["messages"] if getattr(m, "usage_metadata", None)]
print(f"⏱ {duree:.0f} s | outils appelés : {appels}")
print(f"jetons d'entrée par appel au modèle : {jetons}")
print("plan (todos) :", etat.get("todos"))
print("fichiers virtuels :", list((etat.get("files") or {}).keys()))
print("\n🤖", etat["messages"][-1].content)
langfuse.flush()

**👀 Ce que vous devez observer** (mesuré pendant la préparation, poste CPU) :
- **15 à 20 minutes** pour cette question (deux exécutions), contre ~4 minutes pour le coordinateur fait main de J3-02 ;
- **2 500 à 3 400 jetons d'entrée** par appel au modèle principal : le harnais envoie ses consignes et la description de dix outils avant même votre question ;
- l'agent a délégué au sous-agent `diagnostic` (outil `task`)… puis répondu qu'il « n'avait pas de moyen direct de tester un ping » et proposé une commande `ping` à lancer soi-même : le résultat du sous-agent n'a pas été exploité. Pas de plan (`todos` vide : option désactivée par défaut), aucun fichier virtuel ;
- dans Langfuse, la trace `deep-agent` est **beaucoup** plus longue que celle du coordinateur.

> 💡 **Le « Tips » du Formateur :** sur le poste CPU, lancez cette cellule **avant** la pause, ou montrez la trace Langfuse enregistrée. Sur les GPU de la salle, comptez quelques minutes.

**Quand choisir le harnais ?** Pour des tâches **longues et ouvertes** (recherche, rédaction de rapports, analyse de dossiers) avec un modèle capable. Pour un help-desk aux tâches courtes et cadrées, sur un modèle local, le graphe fait main reste plus prévisible, plus rapide et plus lisible.

## 3. Le jalon J3 : `helpdesk_ops.jalons.j3`

```
garde_fou (J1) ─SUPPORT─→ coordinateur ─┬─ demander_kb          (RAG hybride Qdrant, public seulement)
                                         ├─ demander_diagnostic  (statut, ping + MCP inventaire en liste blanche)
                                         ├─ demander_ticketing   (P2 à P4 ; P1 → circuit de validation J2)
                                         └─ demander_acces       (annuaire + KB « acces » : corrigé Ex3.1)
```

In [ ]:
from langchain.mcp import MCPAdapter

import helpdesk_ops.mcp_inventaire as inventaire
from helpdesk_ops.jalons.j3 import construire_helpdesk_j3
from helpdesk_ops.ticketing import Ticketing

outils_mcp = inventaire.filtrer_outils(await MCPAdapter(inventaire.mcp).list_tools(),
                                       autorises={"detail_poste", "historique_interventions"})
helpdesk = construire_helpdesk_j3(llm, outils_mcp=outils_mcp, ticketing=Ticketing())


async def demander(question: str) -> None:
    debut = time.perf_counter()
    with propagate_attributes(session_id="j3-05-jalon", user_id="collab-5aa3b8", tags=["jour-3", "jalon-j3"]):
        etat = await helpdesk.ainvoke({"messages": [("user", question)]},
                                      config={"callbacks": [langfuse_handler], "run_name": "helpdesk-ops-j3", "recursion_limit": 30})
    appels = [c["name"] for m in etat["messages"] for c in (getattr(m, "tool_calls", None) or [])]
    print(f"⏱ {time.perf_counter() - debut:.0f} s | {appels}\n❓ {question}\n🤖 {etat['messages'][-1].content}\n")


await demander("Le PC-0203 est très lent et n'arrive plus à installer ses mises à jour : que faut-il faire ?")
await demander("Quelle est la recette de la tarte au sucre ?")
await demander("Écris-moi un poème sur les serveurs")
langfuse.flush()

**👀 Ce que vous devez observer :**
- la première question mobilise le Diagnostic (outils MCP) et la KB (KB-010, KB-011) ;
- la tarte au sucre est refusée par le garde-fou de J1 **sans aucun appel LLM** ;
- le poème **sur les serveurs** passe le garde-fou (le mot « serveur » est un mot-clé support) : le coordinateur écrit le poème. Contournement par reformulation, déjà vu en Ex1.2 : on le traitera au red-team du J4 ;
- dans Langfuse, **Sessions → `j3-05-jalon`** : la trace de la première question montre le coordinateur, puis les spécialistes imbriqués.

> 🚨 **Vu en préparant ce notebook, et c'est le plus important de la journée :** pour le PC-0203, la réponse attribuait à `[KB-010]` des commandes d'administration (`takeown`, `icacls` sur `C:\Windows\SoftwareDistribution`) à lancer « en tant qu'administrateur ». Or KB-010 dit exactement l'inverse : *ne pas tenter de modifier les droits système, ouvrir un ticket*. Une référence citée, un ton assuré, et un conseil **dangereux** qui contredit sa source. À l'exécution suivante, plus de commandes, mais toujours le conseil de « corriger les droits d'accès », que KB-010 déconseille : **la même question ne donne pas toujours la même réponse**, et une erreur qui n'apparaît qu'une fois sur deux reste une erreur. C'est ce que mesurera la **fidélité aux sources** en J4, sur un dataset et plusieurs exécutions, et ce que filtre un contrôle de sortie (red-team du J4).

## 4. Ce qu'on retient du jour 3

| Brique | La phrase à retenir |
|---|---|
| Patterns | qui décide de la suite, qui voit quel contexte : sous-agents, handoffs, skills, routeur, workflow |
| RAG | dense **et** BM25, fusion hybride ; IDF et racinisation française vérifiés ; filtres de payload posés par le code |
| Sous-agents | un spécialiste = un outil ; il ne reçoit que la consigne ; ses traces s'imbriquent |
| Parallélisme | `Send` + reducer ; paralléliser ce qui attend ; un GPU local reste une file |
| MCP | une prise standard… et une dépendance à surveiller : liste blanche par nom, flux sortants coupés |
| Harnais | Deep Agents emballe tout cela ; utile pour l'ouvert, coûteux en contexte pour un petit modèle |

### ✅ Quiz du jour

1. Une demande porte sur deux sujets. Routeur ou sous-agents ? Pourquoi ?
2. Pourquoi la recherche hybride bat-elle la recherche dense sur « 0x80070005 » ?
3. Les procédures internes N2 ne doivent jamais atteindre l'assistant N1. Où le garantir ?
4. Trois branches `Send` appellent le même serveur de modèles. À quoi s'attendre ?
5. Un serveur MCP tiers déclare tous ses outils `readOnlyHint`. Suffit-il de filtrer sur cette annotation ?
6. Que faut-il toujours passer à `create_deep_agent`, et pourquoi ?

<details>
<summary>Réponses</summary>

1. Sous-agents : le coordinateur délègue à chaque spécialiste concerné puis synthétise ; le routeur n'en choisit qu'un.
2. Le code exact est un terme rare sans « sens » pour l'embedding ; BM25 le trouve, la fusion le place en tête.
3. Dans le code de recherche : un filtre de payload `visibilite = public` (et des index), jamais dans le prompt.
4. Un gain inférieur à ×3 : les requêtes partagent le GPU (lots avec vLLM, file avec Ollama) ; paralléliser les sources, garder une synthèse LLM unique.
5. Non : une annotation est une déclaration ; il faut une liste blanche par nom, après lecture du code.
6. `model=` explicitement : sinon le harnais utilise par défaut un modèle cloud.
</details>

**Tout de suite, J4-01 :** l'observabilité LLM, en relisant ces trois jours de traces. **Demain (J4) :** évaluation avec Langfuse, puis red-team et mini-projet. Ce qu'on a observé « à l'œil » aujourd'hui (le ticket inventé, les catégories approximatives, l'article hors sujet) devient **mesurable** : scores, datasets, experiments.